In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import os

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

 
from src.feature_selection import feature_importance, select_top_features

In [11]:
df = pd.read_csv("../data/KDDTrain+.txt", header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,33,34,35,36,37,38,39,40,41,42
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [12]:
column_names = [
"duration","protocol_type","service","flag","src_bytes","dst_bytes",
"land","wrong_fragment","urgent","hot","num_failed_logins",
"logged_in","num_compromised","root_shell","su_attempted",
"num_root","num_file_creations","num_shells","num_access_files",
"num_outbound_cmds","is_host_login","is_guest_login","count",
"srv_count","serror_rate","srv_serror_rate","rerror_rate",
"srv_rerror_rate","same_srv_rate","diff_srv_rate",
"srv_diff_host_rate","dst_host_count","dst_host_srv_count",
"dst_host_same_srv_rate","dst_host_diff_srv_rate",
"dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
"dst_host_serror_rate","dst_host_srv_serror_rate",
"dst_host_rerror_rate","dst_host_srv_rerror_rate",
"label","difficulty"
]

df.columns = column_names
df.head(25)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21
5,0,tcp,private,REJ,0,0,0,0,0,0,...,0.07,0.07,0.00,0.00,0.00,0.00,1.00,1.00,neptune,21
6,0,tcp,private,S0,0,0,0,0,0,0,...,0.04,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,21
7,0,tcp,private,S0,0,0,0,0,0,0,...,0.06,0.07,0.00,0.00,1.00,1.00,0.00,0.00,neptune,21
8,0,tcp,remote_job,S0,0,0,0,0,0,0,...,0.09,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,21
9,0,tcp,private,S0,0,0,0,0,0,0,...,0.05,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,21


In [13]:
print(df["protocol_type"].unique())
print(df["service"].unique())
print(df["flag"].unique())

['tcp' 'udp' 'icmp']
['ftp_data' 'other' 'private' 'http' 'remote_job' 'name' 'netbios_ns'
 'eco_i' 'mtp' 'telnet' 'finger' 'domain_u' 'supdup' 'uucp_path' 'Z39_50'
 'smtp' 'csnet_ns' 'uucp' 'netbios_dgm' 'urp_i' 'auth' 'domain' 'ftp'
 'bgp' 'ldap' 'ecr_i' 'gopher' 'vmnet' 'systat' 'http_443' 'efs' 'whois'
 'imap4' 'iso_tsap' 'echo' 'klogin' 'link' 'sunrpc' 'login' 'kshell'
 'sql_net' 'time' 'hostnames' 'exec' 'ntp_u' 'discard' 'nntp' 'courier'
 'ctf' 'ssh' 'daytime' 'shell' 'netstat' 'pop_3' 'nnsp' 'IRC' 'pop_2'
 'printer' 'tim_i' 'pm_dump' 'red_i' 'netbios_ssn' 'rje' 'X11' 'urh_i'
 'http_8001' 'aol' 'http_2784' 'tftp_u' 'harvest']
['SF' 'S0' 'REJ' 'RSTR' 'SH' 'RSTO' 'S1' 'RSTOS0' 'S3' 'S2' 'OTH']


In [14]:
categorical_cols = ["protocol_type","service","flag"]
df = encode_categorical(df, categorical_cols)

In [15]:
y = df["label"]
y = y.apply(lambda x: 0 if x == "normal" else 1)

# Drop the label and difficulty columns to create our features dataset
X = df.drop(columns=["label", "difficulty"])

In [16]:
importance = feature_importance(X, y)
importance.head(20)

flag_SF                        0.110212
same_srv_rate                  0.085944
dst_host_same_srv_rate         0.067555
dst_host_srv_count             0.065565
logged_in                      0.064962
dst_host_serror_rate           0.045825
srv_serror_rate                0.045727
serror_rate                    0.042474
dst_host_srv_serror_rate       0.038574
protocol_type_icmp             0.034618
service_private                0.034216
service_http                   0.030492
dst_host_same_src_port_rate    0.026019
src_bytes                      0.023832
flag_S0                        0.022735
dst_host_count                 0.017830
rerror_rate                    0.016902
dst_host_srv_rerror_rate       0.016458
service_eco_i                  0.015678
service_ecr_i                  0.015279
dtype: float64

In [17]:
X_selected = select_top_features(X, importance, top_n=20)

In [18]:
# Split the selected features into training and testing sets
X_train, X_test, y_train, y_test = split_data(X_selected, y, test_size=0.2)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (100778, 20), y_train shape: (100778,)
X_test shape: (25195, 20), y_test shape: (25195,)


In [19]:
# Handle imbalanced data by applying SMOTE on the training set
X_train_res, y_train_res = apply_smote(X_train, y_train)

print(f"Resampled training set shape: {X_train_res.shape}")
print(f"Resampled target shape: {y_train_res.shape}")

Resampled training set shape: (107842, 20)
Resampled target shape: (107842,)


In [20]:
# Scale features (Fit on training, transform both)
X_train_scaled, scaler = scale_features(X_train_res)

# Need to scale test set using the SAME scaler returned from the training step
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

X_train_scaled shape: (107842, 20)
X_test_scaled shape: (25195, 20)


In [21]:
# Save the processed data for modeling usage
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(X_train_scaled).to_csv(processed_dir / "X_train.csv", index=False)
pd.DataFrame(X_test_scaled).to_csv(processed_dir / "X_test.csv", index=False)
y_train_res.to_csv(processed_dir / "y_train.csv", index=False)
y_test.to_csv(processed_dir / "y_test.csv", index=False)

print("Data saved successfully to /data/processed/")

Data saved successfully to /data/processed/
